# <u>**Feature Engineering**</u>

The cleaned dataset is engineered into a set of meaningful and model-ready features that can improve machine learning performance predicting patterns in the National Lottery Project Grants.

EDA shows the <code>award_amount</code> distribution is positively skewed and long‑tailed with meaningful variations between artforms, regions, and strands. 

These characteristics make a multi-class model such as a Descion Tree Classifier or Random Forest more compatible than a linear regression.  

</b>

## <u>**Load Libraries and Data**</u>

Libraries - Pandas and Numpy - are imported. Cleaned data is loaded using Pandas.  

In [4]:
# Import libraries 
import pandas as pd
import numpy as np

# Load DataFrame
df = pd.read_csv('/Users/admin/Documents/Code Institute/ace_project/data/clean/cleaned_data.csv')

# Create copy for feature engineering 

df_fe = df.copy()

</b>

## <u>**Check Data Integrity**</u>

A review of the cleaned dataset is made to confirm the structure, column types and statistical summary. 

In [5]:
# Review the DataFrame
df_fe.info()

# Summary statistics of the DataFrame
df_fe.describe()

<class 'pandas.DataFrame'>
RangeIndex: 5454 entries, 0 to 5453
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   award_amount           5454 non-null   int64
 1   award_date             5454 non-null   str  
 2   decision_month         5454 non-null   str  
 3   decision_quarter       5454 non-null   str  
 4   ace_area               5454 non-null   str  
 5   local_authority        5454 non-null   str  
 6   main_discipline        5454 non-null   str  
 7   strand                 5454 non-null   str  
 8   time_limited_priority  5454 non-null   str  
 9   award_year             5454 non-null   int64
dtypes: int64(2), str(8)
memory usage: 426.2 KB


,award_amount,award_year
count,5.454000e+03,5454.000000
mean,4.068841e+04,2024.189952
std,7.853711e+04,1.161237
min,0.000000e+00,2023.000000
25%,1.939175e+04,2023.000000
50%,2.940650e+04,2024.000000
75%,3.471500e+04,2025.000000
max,2.019149e+06,2026.000000


</b>

## <u>**Flag Zero Awards**</u>

EDA confirms "£0" values are valid awards rather than errors. They represent specific funding decisions. Create a binary flag to treat zero awards as a categorical signal rather than a numerical anomaly. 

However, since these rows don't have an outcome yet it would be misleading to assign them to a tier because it could distort the classification model performance. Hence, a copy of the DataFrame is made to exclude the two rows from modelling.


In [6]:
# Create a flag to identify pending/zero-award records
df_fe["pending_awards"] = (df_fe["award_amount"] == 0).astype(int)


# Check number of rows before filtering
print("Before filtering:", len(df_fe))


# Remove pending/zero-award records
df_fe = df_fe[df_fe["award_amount"] > 0].copy()


# Check number of rows after filtering
print("After filtering:", len(df_fe))


# Remove the temporary flag
df_fe = df_fe.drop(columns=["pending_awards"])


Before filtering: 5454
After filtering: 5452


</b>

## <u>**Award Amount Tiers**</u>

Apply low, medium and high classification tiers to respect the nuances of records in the National Lottery Project Grants data and avoid scaling to manipulate the <code>award_amount</code> to fit a linear regression model. 

The goal isn't simply to make the tiers statistically neat. Random Forest builds many decision trees using samples and features. If one target class is substantially smaller than the others, the model may learn to predict the majority class while perform poorly on the minority class.

Transform the remaining 5,452 valid award records into three target classes using tertile-based thresholds. 

In [7]:
# Calculate the 33rd and 67th percentiles
p33 = df_fe['award_amount'].quantile(0.33)
p67 = df_fe['award_amount'].quantile(0.67)

print("33rd percentile:", p33)
print("67th percentile:", p67)

# Create the three award tiers
df_fe['award_tier'] = pd.cut(
    df_fe['award_amount'],
    bins=[-np.inf, float(p33), float(p67), np.inf],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True
)

# Check the number of observations in each tier
print(df_fe['award_tier'].value_counts())

# Check percentages
print(df_fe['award_tier'].value_counts(normalize=True) * 100)

33rd percentile: 23750.0
67th percentile: 30000.0
award_tier
Medium    1876
Low       1800
High      1776
Name: count, dtype: int64
award_tier
Medium    34.409391
Low       33.015407
High      32.575202
Name: proportion, dtype: float64


The 33rd percentile (£23,750) and 67th percentile (£30,000) were selected as the boundaries, producing highly balanced Low (33.02%), Medium (34.41%) and High (32.58%) classes. This balanced target distribution is suitable for training a Random Forest multi-class classifier because no class is substantially underrepresented.

</b>